In [ ]:
# parameters
# BINDER_FAST: set N=10, n_ecd_cat=2 for fast cloud execution
N = 20            # Hilbert space truncation (Fock dimension)
beta_cat = 2.0 + 0j   # ECD amplitude for cat state preparation
n_ecd_cat = 4     # number of ECD repetitions (unused in single-gate demo,
                   # used in multi-ECD section)


In [ ]:
# hide
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
%matplotlib widget

from bosonic_gates.gates import ecd_operator, ecd_circuit
from bosonic_gates.gates.ecd import ecd_qubit_reset_sequence
from bosonic_gates.states import cat_state, fock_state, coherent_state


## Module 4b: ECD Gates — Echoed Conditional Displacement

**Learning objectives:**
- Understand how a conditional displacement entangles the qubit and cavity
- See how the *echo* (qubit $\pi$ pulse) disentangles the qubit and leaves only
  the cavity displaced
- Prepare a cat state by tracing over the qubit after an ECD circuit
- Build a multi-ECD sequence with interleaved displacements
- Verify the physical decomposition `ecd_qubit_reset_sequence` against the
  full `ecd_operator`

**Tensor ordering:** Throughout this notebook the joint Hilbert space is
**qubit $\otimes$ cavity** (qubit is the *left* factor).  The qubit lives in
$\mathbb{C}^2$ and the cavity in a Fock space of dimension $N$.
All operators returned by `ecd_operator` and `ecd_circuit` share this convention.

*References:*
Campagne-Ibarcq *et al.*, Nature **584**, 368 (2020);
Eickbusch *et al.*, Nature Physics **18**, 1464 (2022).

---

**Sections:**
[1 Conditional Displacement](#sec1) ·
[2 ECD Operator](#sec2) ·
[3 Cat State Preparation](#sec3) ·
[4 Multi-ECD Circuit](#sec4) ·
[5 Physical Decomposition](#sec5) ·
[Exercises](#exercises)


<a id="sec1"></a>
## 1  The Conditional Displacement

### 1.1  What is a conditional displacement?

In a dispersive qubit-cavity system a microwave drive at the cavity frequency
displaces the cavity by an amount that *depends on the qubit state*.  The
resulting operation on the joint space is:

$$\hat{D}_{\rm cond}(\beta) = \hat{D}(+\beta/2)\otimes|e\rangle\langle e|
                             + \hat{D}(-\beta/2)\otimes|g\rangle\langle g|.$$

(Tensor order: qubit $\otimes$ cavity.)

Starting from a product state $|+x\rangle_q \otimes |0\rangle_c$
(qubit in $x$-eigenstate $(|g\rangle+|e\rangle)/\sqrt{2}$, cavity in vacuum)
the conditional displacement creates an entangled state:

$$|+x\rangle_q\otimes|0\rangle_c
\;\xrightarrow{\hat{D}_{\rm cond}(\beta)}\;
\frac{1}{\sqrt{2}}\Bigl(|e\rangle|{+\beta/2}\rangle + |g\rangle|{-\beta/2}\rangle\Bigr).$$

### 1.2  The echo: decoupling the qubit

The joint state above is entangled — the qubit and cavity cannot be separated.
To restore the qubit we apply a $\pi$ pulse ($\hat{X}_q$) on the qubit,
then a second conditional displacement $\hat{D}_{\rm cond}(-\beta)$.  The full
sequence — two conditional displacements with an interleaved $\pi$ pulse — is
the **Echoed Conditional Displacement (ECD)**:

$$\text{ECD}(\beta) = \hat{D}(+\beta/2)\otimes|e\rangle\langle e|
                    + \hat{D}(-\beta/2)\otimes|g\rangle\langle g|.$$

After ECD, the qubit returns to its initial state and the cavity is displaced
conditionally.  The net action on the *cavity* is a superposition of
$\hat{D}(\pm\beta/2)$ conditioned on the qubit measurement outcome.

---
**Lab note.** *The echo removes the undesired qubit rotation acquired during
the conditional drive.  Without it the qubit would end up entangled with the
cavity after every gate, making state preparation impossible.  The physical
implementation uses a driven dispersive interaction at $\omega_c \pm \chi/2$,
sandwiched by transmon $\pi$ pulses.*


<a id="sec2"></a>
## 2  The ECD Operator

The function `ecd_operator(N, beta)` returns the $2N \times 2N$ unitary

$$\text{ECD}(\beta) = |e\rangle\langle e|\otimes\hat{D}(+\beta/2)
                    + |g\rangle\langle g|\otimes\hat{D}(-\beta/2)$$

acting on the joint qubit $\otimes$ cavity space.


In [ ]:
beta_demo = 1.0 + 0j
U_ecd = ecd_operator(N, beta_demo)

print("ECD operator shape:", U_ecd.shape)
print("Expected: (", 2*N, ",", 2*N, ")")

# Verify unitarity
I_joint = qt.tensor(qt.qeye(2), qt.qeye(N))
diff = (U_ecd.dag() * U_ecd - I_joint).norm()
print(f"Unitarity ||U†U - I|| = {diff:.2e}  (should be ~0)")


In [ ]:
# Apply ECD to |+x>_q ⊗ |0>_c  and trace out qubit

# Qubit |+x> = (|g> + |e>) / sqrt(2)
g = qt.basis(2, 0)
e = qt.basis(2, 1)
plus_x = (g + e).unit()

# Joint initial state
psi0 = qt.tensor(plus_x, qt.basis(N, 0))

# Apply ECD
psi_after = U_ecd * psi0

# Trace out qubit (subsystem 0), get cavity state
rho_joint  = qt.ket2dm(psi_after)
rho_cavity = rho_joint.ptrace(1)   # cavity is the second (right) subsystem

print("Cavity state after ECD, purity:", qt.entropy_vn(rho_cavity, base=2))
print("(Entropy > 0 means the cavity is entangled with the qubit)")

# The cavity partial trace is a mixture of two displaced vacua
xvec = np.linspace(-4, 4, 100)
W_cav = qt.wigner(rho_cavity, xvec, xvec)
vmax = np.max(np.abs(W_cav))

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.contourf(xvec, xvec, W_cav, levels=50, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
ax.set_xlabel(r"$X = \mathrm{Re}(\alpha)$")
ax.set_ylabel(r"$P = \mathrm{Im}(\alpha)$")
ax.set_title(rf"Wigner of cavity after ECD($\beta={beta_demo:.1f}$), qubit traced out")
ax.set_aspect("equal")
# Annotate the two coherent blobs
ax.axvline(np.real(beta_demo)/2,  ls="--", color="k", lw=1,
           label=rf"$+\beta/2 = {np.real(beta_demo)/2:.2f}$")
ax.axvline(-np.real(beta_demo)/2, ls="-.",  color="k", lw=1,
           label=rf"$-\beta/2 = {-np.real(beta_demo)/2:.2f}$")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


**Observation.** The Wigner function of the partial trace shows two displaced
blobs at $\pm\beta/2$ without interference fringes — the cavity is in a
*mixed* state because it is still entangled with the qubit.  A subsequent
qubit measurement projects the cavity into a pure displaced coherent state
(either $|{+\beta/2}\rangle$ or $|{-\beta/2}\rangle$).  To create a
*superposition* (cat state) we use a Hadamard before ECD.


<a id="sec3"></a>
## 3  Cat State Preparation via ECD

The standard recipe for a cat state uses a single ECD gate after preparing
the qubit in $|+x\rangle$:

$$|0\rangle_q\otimes|0\rangle_c
\xrightarrow{H_q}
|+x\rangle_q\otimes|0\rangle_c
\xrightarrow{\text{ECD}(\beta)}
\frac{1}{\sqrt{2}}\Bigl(|e\rangle|{+\beta/2}\rangle+|g\rangle|{-\beta/2}\rangle\Bigr)
\xrightarrow{\text{measure }|g\rangle_q}
|{-\beta/2}\rangle_c.$$

That is a *coherent* state, not a cat.  To get a cat state we must measure in
the $x$-basis instead:

$$\xrightarrow{\text{measure }|+x\rangle_q}
\frac{|{+\beta/2}\rangle + |{-\beta/2}\rangle}{\sqrt{2}} = |\mathcal{C}^+_{\beta/2}\rangle.$$

In practice, after ECD we apply a second Hadamard and measure in the
computational basis.  The $|g\rangle$ outcome gives the even cat; $|e\rangle$
gives the odd cat.  Below we project onto $|+x\rangle$ analytically.


In [ ]:
# Step-by-step cat state preparation

# 1. Prepare |0>_q ⊗ |0>_c
psi_g0 = qt.tensor(qt.basis(2, 0), qt.basis(N, 0))

# 2. Hadamard on qubit: |0>_q → |+x>_q
H_q = qt.hadamard_transform(1)  # 2x2 Hadamard
psi_after_H = qt.tensor(H_q, qt.qeye(N)) * psi_g0
print("After Hadamard, qubit state:",
      qt.ket2dm(psi_after_H).ptrace(0).full().round(4))

# 3. Apply ECD(beta_cat)
U_ecd_cat = ecd_operator(N, beta_cat)
psi_entangled = U_ecd_cat * psi_after_H

# 4. Project qubit onto |+x> = (|g>+|e>)/sqrt(2)
plus_x_q = (qt.basis(2, 0) + qt.basis(2, 1)).unit()
# Projection: <+x|_q psi_entangled>  — result lives in cavity space
proj = qt.tensor(plus_x_q.dag(), qt.qeye(N))
psi_cat_unnorm = proj * psi_entangled
psi_cat = psi_cat_unnorm.unit()

prob = float((psi_cat_unnorm.dag() * psi_cat_unnorm).tr().real)
print(f"Probability of |+x> outcome: {prob:.4f}  (expected 0.5)")
print("Cavity cat state purity:",
      qt.entropy_vn(qt.ket2dm(psi_cat), base=2),
      "(0 = pure)")


In [ ]:
# Compare to cat_state() from the library
alpha_cat_half = np.abs(beta_cat) / 2  # cat amplitude = beta/2
psi_cat_ref = cat_state(alpha=alpha_cat_half, N=N, phase=0).state  # even cat

fidelity = abs(float(psi_cat_ref.dag() * psi_cat))**2
print(f"Fidelity with reference cat_state(alpha={alpha_cat_half}): {fidelity:.6f}")
# Note: fidelity may be < 1 due to global phase differences; |overlap|^2 is used


In [ ]:
# Wigner function of the ECD-prepared cat state
xvec = np.linspace(-5, 5, 120)
W_cat_ecd = qt.wigner(qt.ket2dm(psi_cat),     xvec, xvec)
W_cat_ref = qt.wigner(qt.ket2dm(psi_cat_ref), xvec, xvec)

vmax = max(np.max(np.abs(W_cat_ecd)), np.max(np.abs(W_cat_ref)))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, W, title in zip(axes,
                         [W_cat_ecd, W_cat_ref],
                         [rf"ECD-prepared cat ($\beta = {np.real(beta_cat):.1f}$)",
                          rf"Reference even cat ($\alpha = {alpha_cat_half:.2f}$)"]):
    ax.contourf(xvec, xvec, W, levels=60, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel(r"$X$")
    ax.set_ylabel(r"$P$")
    ax.set_aspect("equal")

plt.suptitle("Cat state Wigner function: ECD vs reference", fontsize=12)
plt.tight_layout()
plt.show()

print("Interference fringe visibility: both plots should show negative regions")
print(f"Min W (ECD):  {np.min(W_cat_ecd):.3f}")
print(f"Min W (ref):  {np.min(W_cat_ref):.3f}")


**Observation.** Both Wigner functions show two coherent-state blobs at
$\pm\beta/2$ connected by oscillatory interference fringes — the hallmark of
a cat state.  The negative values in the fringes are a signature of
non-classicality.  A larger $|\beta|$ pushes the blobs further apart, making
the fringes finer and the state more fragile to photon loss.

---
**Lab note.** *In experiment the qubit measurement is done in the $x$-basis by
applying a $\pi/2$ pulse before measurement.  The outcome selects the even or
odd cat.  Repeating the procedure deterministically (by conditioning on the
measurement outcome) allows reliable cat state preparation.*


<a id="sec4"></a>
## 4  Multi-ECD Circuit

More complex states can be prepared by concatenating multiple ECD gates,
optionally interleaved with unconditional cavity displacements $\hat{D}(\alpha_k)$.  The circuit structure is:

$$U = D(\alpha_{K+1})\;\text{ECD}(\beta_K)\;D(\alpha_K)\;\cdots\;D(\alpha_1)\;\text{ECD}(\beta_1)\;D(\alpha_0)$$

where $K$ is the number of ECD gates and the $\alpha_k$ are optional
displacement amplitudes with `len(alphas) = len(betas) + 1`.

This is the basic building block of the GKP state preparation protocol from
Campagne-Ibarcq *et al.* (2020).


In [ ]:
# Verify: ecd_circuit with a single beta equals ecd_operator
beta_single = 0.8 + 0.3j
U_circuit_1 = ecd_circuit(N, [beta_single])
U_direct_1  = ecd_operator(N, beta_single)

diff_single = (U_circuit_1 - U_direct_1).norm()
print(f"Single ECD: ||ecd_circuit - ecd_operator|| = {diff_single:.2e}")


In [ ]:
# Multi-ECD circuit: two ECD gates with interleaved displacements
betas_multi  = [beta_cat, -beta_cat * 0.5]
alphas_multi = [0.0, beta_cat * 0.2, 0.0]   # len = len(betas) + 1

U_multi = ecd_circuit(N, betas_multi, alphas_multi)
print("Multi-ECD circuit shape:", U_multi.shape)

# Apply to |+x>_q ⊗ |0>_c and visualise the cavity state
psi_init = qt.tensor((qt.basis(2,0)+qt.basis(2,1)).unit(), qt.basis(N, 0))
psi_final = U_multi * psi_init

rho_cav_multi = qt.ket2dm(psi_final).ptrace(1)

xvec = np.linspace(-5, 5, 100)
W_multi = qt.wigner(rho_cav_multi, xvec, xvec)
vmax = np.max(np.abs(W_multi))

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.contourf(xvec, xvec, W_multi, levels=50, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
ax.set_title("Cavity Wigner after multi-ECD circuit (qubit traced out)")
ax.set_xlabel(r"$X$")
ax.set_ylabel(r"$P$")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

print("Cavity purity (von Neumann entropy):",
      qt.entropy_vn(rho_cav_multi, base=2))


<a id="sec5"></a>
## 5  Physical Decomposition: `ecd_qubit_reset_sequence`

The ECD gate is physically implemented as a sequence of **four** operations:

1. **Conditional displacement $\hat{D}_{cond}(+\beta/2)$** — drives the cavity
   conditioned on the qubit being in $|e\rangle$.
2. **Qubit $\pi$ pulse** ($\hat{X}$ gate) — flips $|g\rangle\leftrightarrow|e\rangle$.
3. **Conditional displacement $\hat{D}_{cond}(-\beta/2)$** — second half of
   the echo.
4. **Qubit $\pi$ pulse** ($\hat{X}$ gate) — restores the qubit to its initial state.

The two $\pi$ pulses cancel for the qubit, while the two conditional
displacements add constructively on the cavity, realising
$\hat{D}(+\beta/2)|e\rangle + \hat{D}(-\beta/2)|g\rangle$.


In [ ]:
# Verify: product of the 4-step decomposition equals ecd_operator
beta_verify = 1.2 + 0.4j
ops = ecd_qubit_reset_sequence(N, beta_verify)

print(f"Number of operations in decomposition: {len(ops)}")
for i, op in enumerate(ops, 1):
    print(f"  Step {i}: shape = {op.shape}")

# Compose: ops are applied in order, so U_total = ops[-1] * ... * ops[0]
U_composed = ops[0]
for op in ops[1:]:
    U_composed = op * U_composed

U_direct = ecd_operator(N, beta_verify)

# Compare up to global phase: |Tr(U_composed† U_direct)| / dim = 1
d = 2 * N
fid_trace = abs((U_composed.dag() * U_direct).tr()) / d
print(f"\nTrace fidelity |Tr(U†V)|/d = {fid_trace:.8f}  (should be 1.0)")

# Element-wise comparison (global phase may differ)
phase = (U_direct * U_composed.dag()).tr() / d  # global phase factor
diff_mat = (U_composed * phase - U_direct).norm()
print(f"||U_composed - U_direct|| (after phase alignment) = {diff_mat:.2e}")


In [ ]:
# Confirm qubit returns to its initial state after the sequence
# Start with qubit in |g>, cavity in vacuum
psi_g_vac = qt.tensor(qt.basis(2, 0), qt.basis(N, 0))

psi_after_decomp = psi_g_vac
for op in ops:
    psi_after_decomp = op * psi_after_decomp

# Compare qubit state before and after
rho_q_before = qt.ket2dm(psi_g_vac).ptrace(0)
rho_q_after  = qt.ket2dm(psi_after_decomp).ptrace(0)
diff_qubit = (rho_q_before - rho_q_after).norm()
print(f"Qubit state change after ECD decomposition: {diff_qubit:.2e}  (should be ~0)")

# Compare cavity state to ecd_operator result
psi_via_op = ecd_operator(N, beta_verify) * psi_g_vac
rho_cav_decomp = qt.ket2dm(psi_after_decomp).ptrace(1)
rho_cav_op     = qt.ket2dm(psi_via_op).ptrace(1)
fid_cav = qt.fidelity(rho_cav_decomp, rho_cav_op)
print(f"Cavity fidelity decomp vs operator: {fid_cav:.8f}  (should be ~1)")


---
**Lab note.** *In experiment the two $\pi$ pulses are Gaussian-shaped
$\sim 20\,\text{ns}$ transmon pulses.  The conditional displacements are longer
($\sim 100\,\text{ns}$ each) and require precise calibration of the drive
frequency and amplitude to achieve the target $\beta$ amplitude.  The
decomposition above assumes ideal pulses; in reality, Kerr nonlinearity
during the conditional drive introduces small corrections.*


<a id="exercises"></a>
## Exercises

**Exercise 1.** Prepare an **odd** cat state using ECD.  Recall that the odd
cat has the qubit measurement outcome $|e\rangle$ in the $x$-basis (project
onto $|-x\rangle = (|g\rangle - |e\rangle)/\sqrt{2}$).  Compute and plot its
Wigner function.  Compare to `cat_state(alpha, N, phase=np.pi)`.


In [ ]:
# Exercise 1
# YOUR CODE HERE


**Exercise 2.** Verify that $\text{ECD}(\beta=0)$ is the identity on the joint
space.  Compute `ecd_operator(N, 0)` and check that it equals
`qt.tensor(qt.qeye(2), qt.qeye(N))`.


In [ ]:
# Exercise 2
# YOUR CODE HERE


**Exercise 3.** What is the Wigner function of
$\text{ECD}(\beta)|0\rangle_c|g\rangle_q$ after projecting the qubit onto
$|g\rangle$?  (That is, what displacement does the cavity acquire?)
Verify your answer analytically and numerically.


In [ ]:
# Exercise 3
# YOUR CODE HERE


In [ ]:
# Solution — Exercise 1: odd cat state
psi_g0_s = qt.tensor(qt.basis(2, 0), qt.basis(N, 0))
H_q_s = qt.hadamard_transform(1)
psi_plus_x = qt.tensor(H_q_s, qt.qeye(N)) * psi_g0_s

U_ecd_s = ecd_operator(N, beta_cat)
psi_ent_s = U_ecd_s * psi_plus_x

# Project onto |-x> = (|g> - |e>)/sqrt(2)
minus_x_q = (qt.basis(2, 0) - qt.basis(2, 1)).unit()
proj_m = qt.tensor(minus_x_q.dag(), qt.qeye(N))
psi_odd_unnorm = proj_m * psi_ent_s
psi_odd = psi_odd_unnorm.unit()

psi_odd_ref = cat_state(alpha=np.abs(beta_cat)/2, N=N, phase=np.pi).state
fid_odd = abs(float(psi_odd_ref.dag() * psi_odd))**2
print(f"Fidelity odd cat (ECD) vs reference: {fid_odd:.6f}")

xvec_s = np.linspace(-5, 5, 100)
W_odd = qt.wigner(qt.ket2dm(psi_odd), xvec_s, xvec_s)
vmax_s = np.max(np.abs(W_odd))

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.contourf(xvec_s, xvec_s, W_odd, levels=50, cmap="RdBu_r", vmin=-vmax_s, vmax=vmax_s)
ax.set_title(r"Odd cat state $|\mathcal{C}^-_{\beta/2}\rangle$ from ECD")
ax.set_xlabel(r"$X$")
ax.set_ylabel(r"$P$")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()


In [ ]:
# Solution — Exercise 2: ECD(0) = identity
U_ecd_zero = ecd_operator(N, beta=0.0+0j)
I_joint = qt.tensor(qt.qeye(2), qt.qeye(N))
diff_zero = (U_ecd_zero - I_joint).norm()
print(f"||ECD(0) - I|| = {diff_zero:.2e}  (should be ~0)")


In [ ]:
# Solution — Exercise 3: project qubit onto |g> after ECD(beta)|g,0>
# ECD(beta)|g,0> = D(+beta/2)|e,0> * 0 + D(-beta/2)|g,0>  (|e><e| term vanishes)
# => after projection onto |g>: cavity = D(-beta/2)|0> = coherent state at -beta/2

beta_ex3 = 1.5 + 0j
psi_g_vac_3 = qt.tensor(qt.basis(2, 0), qt.basis(N, 0))
psi_after_ecd3 = ecd_operator(N, beta_ex3) * psi_g_vac_3

# Project onto |g>
proj_g = qt.tensor(qt.basis(2, 0).dag(), qt.qeye(N))
psi_cav3_unnorm = proj_g * psi_after_ecd3
psi_cav3 = psi_cav3_unnorm.unit()

# Expected: coherent state at -beta/2
psi_expected3 = qt.coherent(N, -beta_ex3/2)
fid3 = abs(float(psi_expected3.dag() * psi_cav3))**2
print(f"Fidelity with D(-beta/2)|0>: {fid3:.6f}  (should be ~1)")
print(f"Cavity displacement = -beta/2 = {-np.real(beta_ex3)/2:.4f}")
